# Run ARCH-plan python package: planner

Runs the hybrid planner directly from Python, without the REST API or the web app.
To use the web app instead, see `run_planner_ui.ipynb`.

## Before running

1. Install ARCH once by running `./run.sh` from the root of the project.
2. Select the `.venv` environment (created by `run.sh`) as the kernel of this notebook.

## 1 Setup

The planner runs from the `src/` folder of the project: EvoChecker reads its `libs` folder from the working directory and writes its temporary files there.

In [ ]:
import json
import os
from pathlib import Path


def find_project_root(start: Path = Path.cwd()) -> Path:
    """Return the root folder of the project (the one containing pyproject.toml).

    Args:
        start: Folder from which the search starts, moving up through its parents.
            Defaults to the current working directory.

    Returns:
        Path of the project root folder.

    Raises:
        FileNotFoundError: If no parent folder contains pyproject.toml.
    """
    for folder in [start, *start.parents]:
        if (folder / "pyproject.toml").is_file():
            return folder
    raise FileNotFoundError(f"pyproject.toml not found in {start} or its parent folders")


PROJECT_ROOT: Path = find_project_root()
os.chdir(PROJECT_ROOT / "src")
print(f"Working directory: {Path.cwd()}")

## 2 Select the planning problem

Path to the JSON file describing the planning problem. The output folder is created next to this file.

In [ ]:
# uc_agr3.json defines mission constraints (ordering of tasks in "task_graph" and a
# maximum duration), which are checked in section 5.
# example.json is a simpler problem without those constraints.
JSON_FILE: Path = PROJECT_ROOT / "assets" / "planningProblem" / "Test-examples" / "uc_agr3.json"

## 3 Run the hybrid planner

Finds a plan with the task planner, and then runs EvoChecker to obtain the Pareto-optimal solutions under uncertainty.
The number of EvoChecker evaluations and the population size are set in `config.ini`.

In [ ]:
from arch_plan.restapi import planner_service
from arch_plan.restapi.memory_db import PROBLEM_DATABASE
from arch_plan.restapi.models import Problem, Problem2Create


def run_planner(json_file: Path, description: str = "My planning problem") -> Problem:
    """Run the hybrid planner (task planner followed by EvoChecker) on a planning problem.

    Args:
        json_file: Path to the JSON file describing the planning problem.
        description: Free-text description of the problem. Defaults to "My planning problem".

    Returns:
        The solved problem, including its unique id and its status.

    Raises:
        RuntimeError: If the planner does not complete successfully.
    """
    problem: Problem = planner_service.create_and_start_problem(
        Problem2Create(description=description, json_file=str(json_file))
    )
    planner_service.run_hybrid_planner(problem, PROBLEM_DATABASE)
    if problem.status != "completed":
        raise RuntimeError(f"Planner {problem.status}: {problem.error_message}")
    return problem


problem: Problem = run_planner(JSON_FILE, description="Example problem")
print(f"Problem {problem.id}: {problem.status}")

## 4 Read the results

All the generated files (PDDL files, plan, EvoChecker files, Pareto front and set) are saved in the output folder.
`output_plan_results.json` combines the plan and the Pareto-optimal solutions.

In [ ]:
from arch_plan.config.config import PROBLEM_OUTPUT_DIR, PROBLEM_OUTPUT_JSON


def get_output_dir(problem: Problem) -> Path:
    """Return the folder where the files generated for a solved problem are saved.

    Args:
        problem: Problem returned by run_planner.

    Returns:
        Path of the output folder.
    """
    return Path(PROBLEM_OUTPUT_DIR[problem.id])


def load_results(problem: Problem) -> dict:
    """Load the combined plan and Pareto results of a solved problem.

    Args:
        problem: Problem returned by run_planner.

    Returns:
        Dictionary with the keys "plan" (list of plan actions) and "pareto"
        (list of Pareto-optimal solutions, each with its QoS and task retries).
    """
    with open(PROBLEM_OUTPUT_JSON[problem.id], "r") as file:
        return json.load(file)


output_dir: Path = get_output_dir(problem)
results: dict = load_results(problem)
print(f"Output folder: {output_dir}")
print(f"Plan actions: {len(results['plan'])}, Pareto-optimal solutions: {len(results['pareto'])}")

### Plan

In [ ]:
print((output_dir / "plan.txt").read_text())

### Pareto-optimal solutions

Each solution trades off the probability of mission success against the cost, by changing the maximum number of retries per task.

In [ ]:
def print_pareto(results: dict) -> None:
    """Print the probability of success and cost of each Pareto-optimal solution.

    Args:
        results: Dictionary returned by load_results.

    Returns:
        None.
    """
    print(f"{'Solution':>8}  {'P(success)':>10}  {'Cost':>10}")
    for solution in results["pareto"]:
        qos: dict = solution["QoS"]
        print(f"{solution['paretoSolutionId']:>8}  {qos['probabilitySucc']:>10.4f}  {qos['cost']:>10.3f}")


print_pareto(results)

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(output_dir / "plot.png")))

## 5 Check the mission constraints when retries are added

A Pareto-optimal solution gives a maximum number of retries per task, but the plan above does not include them.
This section injects retries into the plan and tests whether the mission constraints still hold.

1. **Inject retries:** take the first Pareto-optimal solution and add the maximum number of retries of the first agent. A retry takes the same time as the original task, so 3 retries add 3 more copies of the task to the agent, right after it.
2. **Constraint programming:** the plan with retries is written as a constraint programming problem in CPMpy (`arch_plan.planning.retry_check`). The start time of each action is a decision variable. Each agent keeps the order of its actions and does one at a time, and actions can be delayed but never start earlier than in the original plan. The solver violates as few mission constraints as possible, and then starts every action as early as possible.
3. **Mission constraints tested:**
   - *ordering*: a task starts after its dependencies in `task_graph` are complete (AND: all of them, OR: at least one). A task is complete when its last attempt ends.
   - *deadline*: every action ends by `max_duration`.
   - *completion*: every task in `tasks_to_be_completed` is in the plan.

The original plan is checked with the same constraints, exactly with the times given by the task planner, to tell apart the violations caused by the retries.

### Inject the retries

In [ ]:
from arch_plan.planning import retry_check


def load_problem(json_file: Path) -> dict:
    """Load the planning problem.

    Args:
        json_file: Path to the JSON file describing the planning problem.

    Returns:
        The planning problem as a dictionary.
    """
    with open(json_file, "r") as file:
        return json.load(file)


def print_retries(agent: str, retries: dict[str, int]) -> None:
    """Print the number of retries injected for each task of an agent.

    Args:
        agent: Id of the agent.
        retries: Dictionary {task id: number of retries}.

    Returns:
        None.
    """
    print(f"Retries injected for agent {agent}:")
    for task_id, number in retries.items():
        print(f"  {task_id}: {number}")


planning_problem: dict = load_problem(JSON_FILE)
original_plan: list[retry_check.Action] = retry_check.load_plan(str(output_dir / "plan.txt"))

pareto_solution: dict = results["pareto"][0]                # first Pareto-optimal solution
first_agent: str = retry_check.get_agents(original_plan)[0]  # first agent of the plan
retries: dict[str, int] = retry_check.get_agent_retries(pareto_solution, original_plan, first_agent)
plan_with_retries: list[retry_check.Action] = retry_check.inject_retries(original_plan, retries)

print_retries(first_agent, retries)
print(f"Actions in the original plan: {len(original_plan)}, with retries: {len(plan_with_retries)}")

### Violations summary

In [ ]:
original_check: retry_check.CheckResult = retry_check.check_plan(original_plan, planning_problem, keep_times=True)
retries_check: retry_check.CheckResult = retry_check.check_plan(plan_with_retries, planning_problem)

print(retry_check.format_violations_summary(original_check, retries_check))

### Rescheduled plan

The plan with the retries and its new times is saved in the output folder as `plan_rescheduled_<Pareto solution id>.txt`, in the same format as `plan.txt` (which is left unchanged). A retry appears as a repeated `dotask` line of the same task.

In [ ]:
rescheduled_file: str = retry_check.save_plan(
    retries_check.actions,
    str(output_dir / f"plan_rescheduled_{pareto_solution['paretoSolutionId']}.txt"),
)
print(f"Saved: {rescheduled_file}\n")
print(Path(rescheduled_file).read_text())

### Gantt charts

Original plan (top) and plan with retries (bottom), on the same time axis. The image is also saved in the output folder as `gantt_retries.png`.

In [ ]:
gantt_file: str = retry_check.save_figure(
    retry_check.plot_gantt_comparison(original_check, retries_check),
    str(output_dir / "gantt_retries.png"),
)
display(Image(filename=gantt_file))